In [2]:
import pandas as pd
import numpy as np
import yaml
import os

# INTRO
This code contains the following steps:
1) we collect per-model class threshold from test metrics dataset using `get_class_thresholds` function 
2) we save thresholds to the `YAML` file to store thresholds if we sure the data is correst 
3) we creates the function that will apply thresholds to the class probabilties per pixel using data stored in YAML

## STEP 1: getting class thresholds

In [83]:
#SETTINGS

directory = '.' #REPLACE WITH YOUR DIRECTORY WITH TEST DATA
alpha = .9 #CONFIDENCE COVERAGE: e.g., 0.9 is for 90% guaranteed that the true value is within the interval
thresholds = {} #PER MODEL THRESHOLDS
all_models_thresholds = {} #ALL MODELS THRESHOLDS TO SAVE TO YAML

In [86]:
# GET ALL MODELS THRESHOLDS
def get_class_thresholds(dir, alpha):
    for file in os.listdir(dir):

        if file.endswith('.csv'):
            filepath = os.path.join(dir, file)
            data = pd.read_csv(filepath)
            cp_source = data[['class_prob_1','class_prob_4', 'class_prob_5', 'class_prob_6', 'class_prob_7','observed_class', 'predicted_class']]
            cp_source = cp_source.rename(columns={'observed_class': 'true_label', 'predicted_class': 'predicted_label'})
            filename = filepath.split('/')[-1]

            for true_label in sorted(cp_source['true_label'].unique()):
                test = cp_source[(cp_source['true_label'] == true_label)&(cp_source['predicted_label'] == true_label)] #slice dataframe for the true label and predicted label
                label = true_label # keep the label
                column_name = 'class_prob_' + str(label) #search for the column with class probabilities 
                probabilities = test[column_name]
                if len(probabilities) == 0: #prevent broken code if there are no probabilities for the class
                    qa = np.nan
                else: 
                    quantile_level = np.ceil((len(probabilities) + 1) * (1 - alpha)) / len(probabilities) #OR SUMPLY 1- alpha
                    qa = probabilities.quantile(quantile_level, interpolation='higher') #calculate quantile based on the formula from the methodology paper considering ceil rounding
                thresholds.update({int(label): round(float(qa),3)}) #saving thredholds for each class lable 
            model = filename.split('_')[2]
            all_models_thresholds.update({model: thresholds.copy()})
    
    return all_models_thresholds     

In [87]:
#THRESHOLDS FOR ALL MODELS
output = get_class_thresholds(directory, alpha)
output

{'SVC': {1: 0.515, 4: 0.848, 5: 0.441, 6: 0.913, 7: 0.675},
 'kNN': {1: 0.525, 4: 0.493, 5: 0.556, 6: 0.731, 7: 0.655},
 'XGB': {1: 0.668, 4: 0.895, 5: 0.705, 6: 0.91, 7: 0.733},
 'RandomForest': {1: 0.439, 4: 0.381, 5: 0.497, 6: nan, 7: 0.601}}

## STEP 2: saving thresholds to YAML

In [91]:
#SETTING UP THE DIRECTORY AND PARAMETERS
dir = directory #REPLACE WITH YOUR DIRECTORY
os.makedirs(dir, exist_ok=True)
yaml_path = os.path.join(dir, 'thresholds.yaml') 

#SAVE THRESHOLDS TO YAML
with open(yaml_path, 'w') as f:
    yaml.dump(output, f, sort_keys=False)

## STEP 3: function to get prediction sets


In [89]:
#OPEN YAML

yaml_path = os.path.join(dir, 'thresholds.yaml') 

with open(yaml_path, 'r') as f:
    prediction_thresholds = yaml.safe_load(f)
prediction_thresholds

{'SVC': {1: 0.515, 4: 0.848, 5: 0.441, 6: 0.913, 7: 0.675},
 'kNN': {1: 0.525, 4: 0.493, 5: 0.556, 6: 0.731, 7: 0.655},
 'XGB': {1: 0.668, 4: 0.895, 5: 0.705, 6: 0.91, 7: 0.733},
 'RandomForest': {1: 0.439, 4: 0.381, 5: 0.497, 6: nan, 7: 0.601}}

In [90]:
def conformal_prediction(dataset: pd.DataFrame, thresholds: dict, cols_with_probs: list):
    # Список классов (из ключей thresholds)
    class_ids = list(thresholds.keys())

    # Соответствующие имена колонок
    class_prob_columns = [f'class_prob_{cls}' for cls in class_ids]

    def get_conformal_prediction_set(row):
        return {
            cls for cls, col in zip(class_ids, class_prob_columns)
            if row[col] >= thresholds[cls]
        }

    # Применим к каждой строке
    conformal_prediction_set = dataset[cols_with_probs].apply(get_conformal_prediction_set, axis=1)
    return conformal_prediction_set

# Пример использования
model_name = 'conformal_predictions_RandomForest_df3_notfiltered.csv'

thresholds = all_models_thresholds['RandomForest']
dataset = pd.read_csv(model_name)
cols_with_probs = [col for col in dataset.columns if col.startswith('class_prob_')]

result = conformal_prediction(dataset, thresholds, cols_with_probs)
dataset['conformal_prediction_set'] = result
dataset.head()

,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B12,...,homogeneity1,homogeneity2,class_prob_1,class_prob_4,class_prob_5,class_prob_6,class_prob_7,observed_class,predicted_class,conformal_prediction_set
0,0.069516,0.110253,0.095290,0.299706,0.497115,0.576840,0.558991,0.612314,0.606325,0.156294,...,0.361872,1.0,0.000000,0.002083,0.018371,0.000000,0.979545,7,7,{7}
1,0.073893,0.117041,0.104581,0.299706,0.497115,0.576840,0.566809,0.612314,0.606325,0.156294,...,0.348026,1.0,0.000000,0.005502,0.016705,0.000000,0.977793,7,7,{7}
2,0.056128,0.098315,0.076059,0.287353,0.508654,0.598846,0.574982,0.654187,0.652126,0.140912,...,0.350092,1.0,0.013796,0.001389,0.023479,0.005093,0.956243,7,7,{7}
3,0.058445,0.104401,0.083838,0.287353,0.508654,0.598846,0.579602,0.654187,0.652126,0.140912,...,0.355409,1.0,0.000000,0.001389,0.012963,0.000000,0.985648,7,7,{7}
4,0.061792,0.108380,0.089023,0.295588,0.505385,0.584957,0.575338,0.639106,0.652126,0.144095,...,0.357091,1.0,0.000000,0.000000,0.005370,0.000000,0.994630,7,7,{7}


In [92]:
cols_with_probs

['class_prob_1',
 'class_prob_4',
 'class_prob_5',
 'class_prob_6',
 'class_prob_7']

In [71]:
mask = dataset['conformal_prediction_set'].apply(lambda x: len(x) > 1)
dataset[mask]

,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B12,...,homogeneity1,homogeneity2,class_prob_1,class_prob_4,class_prob_5,class_prob_6,class_prob_7,observed_class,predicted_class,conformal_prediction_set
2022,0.066426,0.100655,0.101556,0.301176,0.794615,0.918831,0.922175,1.000000,0.735005,0.169024,...,0.575588,1.0,0.447014,0.002222,0.533171,0.000000,0.017593,1,5,"{1, 5}"
2168,0.069258,0.126170,0.119274,0.382647,0.839615,0.986833,0.995558,1.000000,0.817884,0.184583,...,0.391528,1.0,0.448431,0.000000,0.536040,0.000000,0.015528,5,5,"{1, 5}"
2174,0.070803,0.125000,0.115817,0.382647,0.839615,0.986833,1.000000,1.000000,0.817884,0.184583,...,0.387080,1.0,0.448541,0.000000,0.535930,0.000000,0.015528,5,5,"{1, 5}"
2181,0.069516,0.133193,0.120138,0.392647,0.859423,1.000000,1.000000,1.000000,0.824973,0.189710,...,0.333233,1.0,0.452847,0.000000,0.534469,0.000000,0.012684,5,5,"{1, 5}"
2192,0.078270,0.135768,0.124028,0.399412,0.848269,0.983045,0.989872,1.000000,0.824973,0.194130,...,0.338081,1.0,0.471466,0.000000,0.518788,0.000000,0.009746,5,5,"{1, 5}"
2255,0.066684,0.110721,0.129646,0.370588,0.751731,0.883478,0.828358,0.958659,0.773719,0.188119,...,0.346685,1.0,0.462996,0.006667,0.503512,0.000000,0.026825,5,5,"{1, 5}"
2493,0.063337,0.111423,0.105013,0.290588,0.754231,0.945527,0.926084,1.000000,0.781897,0.163013,...,0.369446,1.0,0.478214,0.000000,0.510878,0.000000,0.010907,5,5,"{1, 5}"
2514,0.056128,0.096910,0.100691,0.291765,0.725000,0.912698,0.848259,1.000000,0.816794,0.160184,...,0.292256,1.0,0.441048,0.002020,0.507932,0.000000,0.048999,1,5,"{1, 5}"
2519,0.057158,0.106273,0.100259,0.287059,0.725000,0.912698,0.901564,1.000000,0.816794,0.161245,...,0.275651,1.0,0.444979,0.000000,0.502872,0.000505,0.051644,1,5,"{1, 5}"
2529,0.053553,0.110487,0.089023,0.289412,0.736154,0.931097,0.864961,1.000000,0.790622,0.161245,...,0.327968,1.0,0.457224,0.000000,0.510135,0.000000,0.032641,1,5,"{1, 5}"
